# run_official_valid (v4) — single evaluation of the frozen Model B and agent decision layer on official MURA valid (XR_WRIST)

**How to run:** Runtime → Restart session → Run all, on a **CPU** runtime (the internal check and the official run must use the same device type).

| Cell | What it does |
|---|---|
| 1 | Environment and integrity checks: repository ZIP chosen by commit hash, file fingerprints, frozen threshold, canonical weights, official valid images present (count only) |
| 2 | Manifest and Model B inference; predictions saved before any metric |
| 3 | R0: reproduction of the canonical predictions on internal validation (skipped in MODE "official") |
| 4 | Technical QC and the deterministic agent decision layer (agent_policy v1.1) |
| 5 | Endpoints with patient-cluster bootstrap CIs (threshold fixed, as in notebook 06); summary.json |

**Settings (Cell 1) — the only lines that may be edited:**

- `MODE = "internal"` runs everything on internal validation; it can be repeated freely.
- `MODE = "official"` is the single official run; a second official run is refused.
- `CODE_COMMIT` is the full 40-character hash of the repository commit. Optional for `"internal"` (the newest repository ZIP in DATA is used), required for `"official"`.

In [1]:
# =====================================================================================
# CELL 1 — Environment and integrity checks
# Purpose : prove that everything the run will use is the frozen canon, BEFORE any
#           radiograph is read: code version, agent-layer files, threshold, model
#           weights and input tables.
# Reads   : repository ZIP on Drive (code), DATA/MODEL_B_FINE_TUNING (threshold files,
#           checkpoint parts), three input CSVs (fingerprint only, rows are not parsed).
# Writes  : /content only (unpacked code, rebuilt checkpoint). Nothing is written to Drive.
# Stops   : at the first FAIL, printing what failed.
# Run     : always in a fresh session: Runtime -> Restart session -> Run all.
#           A second run in the same session fails on purpose ("Code imported from the
#           ZIP"), because Python would reuse code already loaded from an older unpack.
# =====================================================================================
import os, re, sys, json, hashlib, zipfile, platform
from datetime import datetime, timezone
from pathlib import Path

# ---- Run settings ---------------------------------------------------------------------
# Only these two lines may be edited.
# MODE "internal": R0 and dry run on internal validation (official valid is never parsed);
#                  can be repeated freely.
# MODE "official": the single official run; allowed once.
# CODE_COMMIT: full 40-character hash of the repository commit to run
#              (GitHub -> Commits -> "Copy full SHA").
#              Optional in MODE "internal": if None, the newest repository ZIP in DATA is used
#              and its commit is recorded in the report. Required in MODE "official".
MODE = "internal"
CODE_COMMIT = None

# ---- Frozen fingerprints (SHA-256) -----------------------------------------------------
# Where each value comes from:
#   checkpoint                      -> configs/model_b_artifact_bundle.json (Git)
#   validation_predictions_mean.csv -> docs/results/checkpoint1/README.md (Git)
#   code files                      -> the repository files at the frozen commit
#   input manifests                 -> computed from the files in DATA; frozen by committing
#                                      this notebook to Git
# Any byte change in a file gives a different value.
SHA = {
    "checkpoint": "63AEDC6BEC1534BDED7C3CA4D0DAF03AFC17BF92166CB7848506A6F3A82DE5B6",
    "validation_predictions_mean.csv": "29FE06B39D9E82021C16EBA4903541B0E0ED79D2956985D1DF5F1260D06CFAF2",
    "wrist_internal_val_clean.csv": "9024090DE20D1048F138041C3C5A134BEDA27DDE41BDF3A986F353728753614A",
    "wrist_official_validation.csv": "762329D2992181EE7AD61141FA5186F8823DB5EDFDFE1AB30B242A0E7772214B",
    "configs/model_b_artifact_bundle.json": "22CB3D5753120D1548EFEF194B7C6FD4B695AB142709CBEC3B964C4C499C6B06",
    "tools/agent_policy.py": "FB956F0C4E556DEA64334B78AE68AC7756F931AEC897250ED9B8546CE8EA1D54",
    "src/runtime/deterministic_safety.py": "2DA0AE3F43432C7EB3A87B8A5AEC87D7D6F7CD9B14E4C4208B1DCB32B9A73A45",
}
THRESHOLD_EXPECTED = 0.263567715883255   # frozen operating threshold of Model B
OFFICIAL_IMAGES_EXPECTED = 659           # PNG radiographs of official MURA valid XR_WRIST


def sha256(path):
    """SHA-256 fingerprint of a file, read as raw bytes in 1 MB chunks (content is not parsed)."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


def check(name, ok, detail=""):
    """Print PASS/FAIL, keep the result for the run report, stop the notebook on FAIL."""
    CHECKS[name] = bool(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {name}  {detail}")
    assert ok, f"Stopped: {name} {detail}"


CHECKS = {}   # every check of this run -> saved in the report
RUN = {"mode": MODE, "started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}

# ---- 1. Drive and the shared DATA folder -----------------------------------------------
# Same search as MURA_Agent_Minimal: the first DATA folder that contains MODEL_B_FINE_TUNING.
# To force a path, set the environment variable MURA_DATA_DIR.
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE = Path("/content/drive")
if os.environ.get("MURA_DATA_DIR"):
    DATA_DIR = Path(os.environ["MURA_DATA_DIR"])
else:
    patterns = ["MyDrive/DATA", "MyDrive/*/DATA", "MyDrive/*/*/DATA", ".shortcut-targets-by-id/*/*/*/DATA"]
    found = [p for pattern in patterns for p in DRIVE.glob(pattern) if (p / "MODEL_B_FINE_TUNING").is_dir()]
    DATA_DIR = found[0] if found else None
check("DATA folder found", DATA_DIR is not None, str(DATA_DIR))
MODEL_B_DIR = DATA_DIR / "MODEL_B_FINE_TUNING"   # canonical Model B bundle files
OUT_ROOT = DATA_DIR / "OFFICIAL_VALID_FINAL"      # every run creates its own new subfolder here

# ---- 2. Mode guard ---------------------------------------------------------------------
# The official run needs the hash of the frozen commit and is allowed only once:
# if any official_*/summary.json already exists, the notebook refuses to run again.
check("MODE is valid", MODE in {"internal", "official"}, MODE)
CODE_COMMIT = CODE_COMMIT.strip().lower() if isinstance(CODE_COMMIT, str) and CODE_COMMIT.strip() else None
check("CODE_COMMIT empty or a 40-character hash",
      CODE_COMMIT is None or re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT) is not None, str(CODE_COMMIT))
if MODE == "official":
    check("Code commit set", CODE_COMMIT is not None)
    previous = sorted(OUT_ROOT.glob("official_*/summary.json"))
    check("No previous official run", not previous, ", ".join(map(str, previous)))

# ---- 3. Repository code ----------------------------------------------------------------
# GitHub writes the commit hash into the ZIP comment, so the comment proves which version
# of the repository is used. Every mura-wrist-ml-pipeline-*.zip in DATA is considered
# (a ZIP of main and a ZIP of one specific commit have different names):
#   CODE_COMMIT set  -> the ZIP whose comment equals CODE_COMMIT; none -> FAIL;
#   CODE_COMMIT None -> the newest ZIP (MODE "internal" only).
# The ZIP is unpacked into a new folder, and we confirm that Python imports the project
# code from that folder (not from copies of .py files on Drive).
def zip_comment(path):
    """Commit hash that GitHub wrote into the ZIP comment ('' if there is none)."""
    with zipfile.ZipFile(path) as archive:
        return archive.comment.decode(errors="ignore").strip()

candidates = [Path(os.environ["MURA_REPO_ZIP"])] if os.environ.get("MURA_REPO_ZIP") else \
    sorted(DATA_DIR.glob("mura-wrist-ml-pipeline-*.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
zip_commits = {p: zip_comment(p) for p in candidates if p.is_file()}
print("Repository ZIPs found (newest first):", {p.name: c[:7] for p, c in zip_commits.items()})
if CODE_COMMIT:
    repo_zip = next((p for p, c in zip_commits.items() if c == CODE_COMMIT), None)
else:
    repo_zip = next(iter(zip_commits), None)
check("Repository ZIP found", repo_zip is not None, f"{repo_zip} (requested commit: {CODE_COMMIT or 'newest ZIP'})")
zip_commit = zip_commits[repo_zip]
check("ZIP comment is a commit hash", re.fullmatch(r"[0-9a-f]{40}", zip_commit) is not None, zip_commit)
with zipfile.ZipFile(repo_zip) as archive:
    unpack_dir = Path("/content") / f"repo_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"
    archive.extractall(unpack_dir)
PROJECT_ROOT = next(unpack_dir.iterdir())
sys.path.insert(0, str(PROJECT_ROOT))
import src.data.transforms as _transforms
check("Code imported from the ZIP", Path(_transforms.__file__).resolve().is_relative_to(PROJECT_ROOT.resolve()),
      _transforms.__file__)

# Fingerprints of the three files that define the frozen configuration:
# the Model B passport (bundle) and the two deterministic agent layers.
for rel in ("configs/model_b_artifact_bundle.json", "tools/agent_policy.py", "src/runtime/deterministic_safety.py"):
    check(f"SHA {rel}", sha256(PROJECT_ROOT / rel) == SHA[rel])

# ---- 4. Frozen threshold ---------------------------------------------------------------
# The same threshold is recorded in three independent places:
#   bundle (Git)                          -> model.threshold
#   selection.json (written by training)  -> threshold_selection.threshold
#   frozen_primary_operating_policy.json  -> threshold (the operating policy used downstream)
# All three must agree, and each must state that the choice was made on internal
# validation only (official valid not used).
bundle = json.loads((PROJECT_ROOT / "configs" / "model_b_artifact_bundle.json").read_text())
selection = json.loads((MODEL_B_DIR / "selection.json").read_text())
policy = json.loads((MODEL_B_DIR / "frozen_primary_operating_policy.json").read_text())
THRESHOLD = float(bundle["model"]["threshold"])
check("Bundle id and governance",
      bundle["bundle_id"] == "checkpoint1-model-b-mean-probability-95-sensitivity"
      and bundle["official_mura_valid_used_for_selection"] is False
      and bundle["model"]["study_aggregation"] == "mean probability")
check("selection.json: validation only",
      selection["selection_data"] == "validation only" and selection["test_used_for_selection"] is False)
check("frozen policy: internal validation, mean",
      policy.get("source_split") == "internal_validation" and policy.get("study_aggregation") == "mean_probability")
thresholds = [THRESHOLD, float(selection["threshold_selection"]["threshold"]), float(policy["threshold"])]
check("Threshold identical in 3 sources", max(abs(t - THRESHOLD_EXPECTED) for t in thresholds) < 1e-12, str(thresholds))

# ---- 5. Canonical Model B weights ------------------------------------------------------
# The 134 MB checkpoint is stored on Drive in 3 parts. The repository script checks every
# part, joins them in /content/model_b and verifies the final SHA-256.
# The ready-made .pt file in MODEL_B_FINE_TUNING (assembled by hand on 08.11) is NOT used.
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from reassemble_model_b_checkpoint import reassemble_checkpoint
artifact = reassemble_checkpoint(MODEL_B_DIR / "checkpoint_parts", Path("/content/model_b"))
check("Checkpoint SHA-256 = bundle",
      artifact["checkpoint_sha256"] == SHA["checkpoint"]
      == bundle["required_files"]["best_resnet18_study_level.pt"]["sha256"].upper(),
      artifact["status"])
CHECKPOINT_PATH = Path(artifact["checkpoint"])

# ---- 6. Input tables = the frozen versions ----------------------------------------------
# Fingerprints only: proves each file is byte-identical to its frozen version.
#   validation_predictions_mean.csv -> R0 reference (Model B on internal validation)
#   wrist_internal_val_clean.csv    -> R0 manifest (691 studies, 1942 images)
#   wrist_official_validation.csv   -> official manifest; parsed only in MODE "official"
INPUTS = {
    "validation_predictions_mean.csv": MODEL_B_DIR / "validation_predictions_mean.csv",
    "wrist_internal_val_clean.csv": DATA_DIR / "PREPROCESSED_WRIST" / "wrist_internal_val_clean.csv",
    "wrist_official_validation.csv": DATA_DIR / "wrist_official_validation.csv",
}
for name, path in INPUTS.items():
    check(f"SHA {name}", path.is_file() and sha256(path) == SHA[name], str(path))

# ---- 7. Official valid images are in place (count only) --------------------------------
# Counts the PNG files under DATA/valid/XR_WRIST without opening the official manifest or
# any image, so a missing file shows up in an internal run, not in the official run.
# macOS "._" files are not radiographs and are not counted.
valid_dir = DATA_DIR / "valid" / "XR_WRIST"
n_valid_images = sum(1 for p in valid_dir.rglob("*.png") if not p.name.startswith("._")) if valid_dir.is_dir() else 0
check("Official valid images present", n_valid_images == OFFICIAL_IMAGES_EXPECTED,
      f"{n_valid_images} expected {OFFICIAL_IMAGES_EXPECTED}")

# ---- 8. Environment record ---------------------------------------------------------------
# Library versions and device go into the report: R0 and the official run must use the
# same device type (CPU).
import numpy, pandas, PIL, sklearn, torch, torchvision
RUN.update({
    "data_dir": str(DATA_DIR), "repo_zip": str(repo_zip), "commit": zip_commit,
    "code_commit_requested": CODE_COMMIT, "official_images_found": n_valid_images,
    "threshold": THRESHOLD, "checkpoint_sha256": artifact["checkpoint_sha256"],
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "torchvision": torchvision.__version__, "pillow": PIL.__version__,
                 "numpy": numpy.__version__, "pandas": pandas.__version__, "sklearn": sklearn.__version__},
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
})
RUN["checks"] = CHECKS
print("\nCELL 1: PASSED", json.dumps({k: RUN[k] for k in ("mode", "commit", "threshold", "device", "versions")}, indent=1))

Mounted at /content/drive
PASS  DATA folder found  /content/drive/MyDrive/DATA
PASS  MODE is valid  internal
PASS  CODE_COMMIT empty or a 40-character hash  None
Repository ZIPs found (newest first): {'mura-wrist-ml-pipeline-main.zip': '3ecbcb9'}
PASS  Repository ZIP found  /content/drive/MyDrive/DATA/mura-wrist-ml-pipeline-main.zip (requested commit: newest ZIP)
PASS  ZIP comment is a commit hash  3ecbcb9212ffd7b392b76c732843a1e78c6bd509
PASS  Code imported from the ZIP  /content/repo_20261110T083357Z/mura-wrist-ml-pipeline-main/src/data/transforms.py
PASS  SHA configs/model_b_artifact_bundle.json  
PASS  SHA tools/agent_policy.py  
PASS  SHA src/runtime/deterministic_safety.py  
PASS  Bundle id and governance  
PASS  selection.json: validation only  
PASS  frozen policy: internal validation, mean  
PASS  Threshold identical in 3 sources  [0.263567715883255, 0.263567715883255, 0.263567715883255]
PASS  Checkpoint SHA-256 = bundle  reassembled_and_verified
PASS  SHA validation_predictio

In [2]:
# =====================================================================================
# CELL 2 — Manifest and Model B inference
# Purpose : compute Model B probabilities with the repository code for every image and
#           every study of the selected set:
#             MODE "internal" -> internal validation (691 studies, 1942 images)
#             MODE "official" -> official MURA valid (237 studies, 659 images)
# Reads   : the input CSV checked in Cell 1, the radiographs under DATA/train or DATA/valid.
# Writes  : a NEW folder DATA/OFFICIAL_VALID_FINAL/<mode>_<UTC>/ with two CSV files.
#           Existing files on Drive are never changed.
# Metrics : none. Predictions are saved and fingerprinted before any metric is computed.
# =====================================================================================
import pandas as pd
import torch
from pathlib import PurePosixPath
from torch.utils.data import DataLoader
from src.data.study_dataset import MURAStudyDataset
from src.data.transforms import build_eval_transform
from src.models.resnet18_classifier import build_resnet18_classifier

EXPECTED = {"internal": (691, 1942, 653), "official": (237, 659, 207)}  # studies, images, patients
SOURCE = INPUTS["wrist_internal_val_clean.csv" if MODE == "internal" else "wrist_official_validation.csv"]

# ---- 1. Manifest in the repository format ----------------------------------------------
# One row per image. study_id = folder path ("train/XR_WRIST/patientN/studyM_label"),
# exactly as in validation_predictions_mean.csv, so R0 can join on it.
# Labels are needed only because the repository dataset class requires them;
# no metric is computed in this cell.
table = pd.read_csv(SOURCE, dtype=str)
table["study_id"] = table["file_name"].map(lambda p: str(PurePosixPath(p).parent))
table["label"] = table["label"].map({"negative": 0, "positive": 1})
check("Labels are negative/positive only", table["label"].notna().all())
table["num_views_in_study"] = table.groupby("study_id")["file_name"].transform("size")
rows = [{"patient_id": r.patient_id, "study_id": r.study_id, "image_path": r.file_name, "label": int(r.label),
         "source_split": r.file_name.split("/")[0], "num_views_in_study": int(r.num_views_in_study)}
        for r in table.itertuples(index=False)]
# Population must match the expected counts exactly; macOS "._" files are not radiographs.
counts = (table["study_id"].nunique(), len(table), table["patient_id"].nunique())
check("Counts = expected", counts == EXPECTED[MODE], f"{counts} expected {EXPECTED[MODE]}")
check("No macOS '._' files", not table["file_name"].str.contains("/\\._").any())
# The set is also confirmed by content: internal validation was carved out of MURA train,
# the official set is MURA valid, so every path must start with the matching folder.
prefix = "valid/XR_WRIST/" if MODE == "official" else "train/XR_WRIST/"
check("All images from the expected MURA folder", table["file_name"].str.startswith(prefix).all(), prefix)

# ---- 2. Leakage guard --------------------------------------------------------------------
# No patient of the evaluated set may appear in the internal training split
# (the split file itself is checked against its recorded fingerprint).
split_path = DATA_DIR / "mura_wrist_internal_split.csv"
check("SHA mura_wrist_internal_split.csv", sha256(split_path) == "AC0D8D67C8BEF6E00C2BD8DB51A9002DD8740670A836FA57BE5623D24C9420BF")
split = pd.read_csv(split_path, dtype=str)
train_patients = set(split.loc[split["internal_split"] == "train", "patient_id"])
check("Internal train patients = 2614", len(train_patients) == 2614, str(len(train_patients)))
check("No patient overlap with internal train", not (set(table["patient_id"]) & train_patients))

# ---- 3. Canonical model and preprocessing -------------------------------------------------
# Weights: the checkpoint rebuilt and verified in Cell 1, loaded strictly (every key must
# match). Preprocessing: build_eval_transform(224) = grayscale, black padding to a square,
# resize to 224, 3 channels, ImageNet normalisation, no augmentation.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
model = build_resnet18_classifier(weights="none")
model.load_state_dict(checkpoint["model_state_dict"])  # strict: every key must match
model.to(DEVICE).eval()
dataset = MURAStudyDataset(rows, dataset_root=DATA_DIR, transform=build_eval_transform(image_size=224))
loader = DataLoader(dataset, batch_size=None, shuffle=False, num_workers=2)

# ---- 4. Inference ----------------------------------------------------------------------
# One study at a time: one logit per image -> sigmoid -> image probability;
# study probability = mean of its image probabilities (the frozen aggregation).
view_rows, study_rows = [], []
with torch.inference_mode():
    for item in loader:
        p = torch.sigmoid(model(torch.stack(item["images"]).to(DEVICE))).cpu().double().tolist()
        view_rows += [{"study_id": item["study_id"], "image_path": path, "abnormal_probability": q}
                      for path, q in zip(item["image_paths"], p)]
        study_rows.append({"patient_id": item["patient_id"], "study_id": item["study_id"], "label": item["label"],
                           "num_views": len(p), "abnormal_probability": sum(p) / len(p)})
views = pd.DataFrame(view_rows)
studies = pd.DataFrame(study_rows)
check("Every study and image predicted", (len(studies), len(views)) == EXPECTED[MODE][:2])

# ---- 5. Save predictions first -------------------------------------------------------------
# The CSV files and their fingerprints are recorded BEFORE any metric is computed.
# Format of study_predictions.csv = the repository contract
# (patient_id, study_id, label, num_views, abnormal_probability).
RUN_DIR = OUT_ROOT / f"{MODE}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
views.to_csv(RUN_DIR / "view_predictions.csv", index=False)
studies.to_csv(RUN_DIR / "study_predictions.csv", index=False)
RUN["run_dir"] = str(RUN_DIR)
RUN["device"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
RUN["predictions_sha256"] = {f: sha256(RUN_DIR / f) for f in ("view_predictions.csv", "study_predictions.csv")}
RUN["predictions_saved_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
print("\nCELL 2: PASSED", RUN["run_dir"], RUN["device"], RUN["predictions_sha256"])

PASS  Labels are negative/positive only  
PASS  Counts = expected  (691, 1942, 653) expected (691, 1942, 653)
PASS  No macOS '._' files  
PASS  All images from the expected MURA folder  train/XR_WRIST/
PASS  SHA mura_wrist_internal_split.csv  
PASS  Internal train patients = 2614  2614
PASS  No patient overlap with internal train  


/content/repo_20261110T083357Z/mura-wrist-ml-pipeline-main/src/models/resnet18_classifier.py:197: RuntimeWarning: Building a randomly initialized ResNet-18. This is suitable for a smoke test but not the required pretrained Phase 1 experiment.
  backbone, build_info = build_resnet18_backbone(


PASS  Every study and image predicted  

CELL 2: PASSED /content/drive/MyDrive/DATA/OFFICIAL_VALID_FINAL/internal_20261110T084208Z cpu {'view_predictions.csv': '142D3EFA90CDFA0C70CD9CFADD863A68760BF55A446CFC4248A3A9A2C9DB977A', 'study_predictions.csv': 'E09732CB6CA1179F25FB78C2B6FCD9AAD15456E6117F4C2089AC3E2AD6C07E9E'}


In [3]:
# =====================================================================================
# CELL 3 — R0: reproduction check on internal validation
# Purpose : prove that Cells 1-2 reproduce the canonical Model B predictions saved at
#           training time (validation_predictions_mean.csv). If they do, model, weights,
#           preprocessing and aggregation in this notebook are the canonical ones.
# Pass    : max |p_new - p_reference| <= 0.001 AND no study changes class, except studies
#           lying within 0.001 of the threshold.
# Writes  : r0_report.json in the run folder (all checks, versions, fingerprints, R0 numbers).
# Runs    : only in MODE "internal"; in MODE "official" the cell is skipped (R0 was done
#           in the internal run with the same code).
# =====================================================================================
if MODE == "official":
    print("CELL 3: skipped in MODE 'official' (R0 belongs to the internal run)")
else:

    # ---- 1. Join our predictions with the reference, study by study -------------------------
    # The same 691 studies, the same labels and view counts are required before comparing.
    reference = pd.read_csv(INPUTS["validation_predictions_mean.csv"])
    r0 = reference.merge(studies, on="study_id", how="outer", suffixes=("_ref", "_new"), indicator=True)
    check("Same 691 studies", (r0["_merge"] == "both").all() and len(r0) == 691)
    check("Same labels and view counts", (r0["label_ref"] == r0["label_new"]).all()
          and (r0["num_views_ref"] == r0["num_views_new"]).all())

    # ---- 2. Compare probabilities and classes ---------------------------------------------
    # "far" = the reference probability is more than 0.001 away from the threshold; only for
    # such studies a class change counts as a failure (tiny numeric noise is expected).
    r0["abs_diff"] = (r0["abnormal_probability_ref"] - r0["abnormal_probability_new"]).abs()
    far = (r0["abnormal_probability_ref"] - THRESHOLD).abs() > 1e-3
    class_flips = int(((r0["abnormal_probability_ref"] >= THRESHOLD) != (r0["abnormal_probability_new"] >= THRESHOLD))[far].sum())
    R0 = {"studies": len(r0), "max_abs_diff": float(r0["abs_diff"].max()), "mean_abs_diff": float(r0["abs_diff"].mean()),
          "class_flips_outside_0.001": class_flips,
          "studies_within_0.001_of_threshold": int((~far).sum())}
    print(json.dumps(R0, indent=1))
    check("R0: max |dp| <= 0.001", R0["max_abs_diff"] <= 1e-3, f'{R0["max_abs_diff"]:.2e}')
    check("R0: no class flips away from threshold", class_flips == 0)

    # ---- 3. Report -------------------------------------------------------------------------
    RUN["r0"] = R0
    (RUN_DIR / "r0_report.json").write_text(json.dumps(RUN, indent=1, default=str))
    print("\nCELL 3 (R0): PASSED ->", RUN_DIR / "r0_report.json")

PASS  Same 691 studies  
PASS  Same labels and view counts  
{
 "studies": 691,
 "max_abs_diff": 1.7881393432617188e-06,
 "mean_abs_diff": 1.6441326934188746e-07,
 "class_flips_outside_0.001": 0,
 "studies_within_0.001_of_threshold": 1
}
PASS  R0: max |dp| <= 0.001  1.79e-06
PASS  R0: no class flips away from threshold  

CELL 3 (R0): PASSED -> /content/drive/MyDrive/DATA/OFFICIAL_VALID_FINAL/internal_20261110T084208Z/r0_report.json


In [4]:
# =====================================================================================
# CELL 4 — Technical QC and the agent decision layer (deterministic; no model, no LLM)
# Purpose : route every study exactly as the frozen agent layer would:
#             1) technical QC of every image   -> deterministic_safety.technical_qc
#             2) anatomy and view evidence     -> deterministic_safety (validate_declared_anatomy,
#                                                 view_evidence)
#             3) decision and its validation   -> agent_policy.select_agent_tools (v1.1)
#           Outcomes: priority_human_review / routine_workflow / SAFETY_BLOCK (QC not PASS or
#           an inconsistent record). Every case still goes to a doctor (requires_human_review).
# Reads   : image files again (bytes for QC); probabilities from Cell 2 (the model is NOT rerun).
# Writes  : study_routes.csv in the run folder (+ its SHA-256 in the report).
# Metrics : none.
# =====================================================================================
import importlib.util
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from src.runtime.deterministic_safety import technical_qc, validate_declared_anatomy, view_evidence

# ---- 1. Frozen policy module (file fingerprint was checked in Cell 1) ----------------------
spec = importlib.util.spec_from_file_location("agent_policy", PROJECT_ROOT / "tools" / "agent_policy.py")
agent_policy = importlib.util.module_from_spec(spec)
spec.loader.exec_module(agent_policy)
check("agent_policy version 1.1", agent_policy.POLICY_VERSION == "1.1", agent_policy.POLICY_VERSION)

# ---- 2. Technical QC of every image -------------------------------------------------------
# Same thresholds as notebook 09: side < 128 px or near-uniform image -> FAIL;
# mean brightness < 5 or > 250 -> WARNING. Files are read in parallel (Drive is slow).
def qc_one(path):
    return technical_qc((DATA_DIR / path).read_bytes(), image_id=path)

with ThreadPoolExecutor(max_workers=8) as pool:
    qc = list(pool.map(qc_one, views["image_path"]))
views["qc_status"] = [q["status"] for q in qc]
views["qc_issues"] = [";".join(q["issues"]) for q in qc]
check("QC result for every image", len(qc) == len(views) and set(views["qc_status"]) <= {"PASS", "WARNING", "FAIL"})

# ---- 3. One agent record per study -> policy decision --------------------------------------
# The cell proposes the action by the frozen rule (priority if class 1 OR the views lie on
# both sides of the threshold); agent_policy re-checks the whole record independently and
# returns SAFETY_BLOCK if anything is inconsistent or QC/anatomy is not PASS/SUPPORTED_WRIST.
SEVERITY = {"PASS": 0, "WARNING": 1, "FAIL": 2}
study_p = studies.set_index("study_id")["abnormal_probability"]
route_rows = []
for study_id, g in views.groupby("study_id", sort=True):
    probs = [float(x) for x in g["abnormal_probability"]]
    p = float(study_p[study_id])
    evidence = view_evidence(probs, THRESHOLD)
    qc_status = max(g["qc_status"], key=SEVERITY.get)
    anatomy = validate_declared_anatomy(study_id.split("/")[1])   # "XR_WRIST" from the path
    predicted = int(p >= THRESHOLD)
    action = "priority_human_review" if predicted == 1 or evidence["threshold_disagreement"] else "routine_workflow"
    record = {
        "study_id": study_id, "workflow_status": "SUCCESS", "qc_status": qc_status,
        "anatomy_status": anatomy["status"], "classification_status": "SUCCESS",
        "classification_performed": True, "classification_valid": True, "classification_modified": False,
        "predicted_class": predicted, "study_probability": p, "classification_threshold": THRESHOLD,
        "agent_action": action, "image_predictions": probs,
        "view_consistency": {
            "status": evidence["status"], "image_count": evidence["view_count"],
            "min_probability": evidence["min_probability"], "max_probability": evidence["max_probability"],
            "mean_probability": evidence["mean_probability"], "probability_range": evidence["probability_range"],
            "probability_std": evidence["probability_std"],
            "threshold_disagreement": evidence["threshold_disagreement"],
            "image_classes": [int(x >= THRESHOLD) for x in probs], "threshold": THRESHOLD,
        },
    }
    decision = agent_policy.select_agent_tools(record, THRESHOLD)
    route_rows.append({
        "study_id": study_id, "qc_status": qc_status,
        "qc_issues": ";".join(sorted({i for s in g["qc_issues"] if s for i in s.split(";")})),
        "anatomy_status": anatomy["status"], "predicted_class": predicted,
        "threshold_disagreement": evidence["threshold_disagreement"], "proposed_action": action,
        "policy_status": decision["policy_status"],
        "route": action if decision["policy_status"] == "SUCCESS" else "SAFETY_BLOCK",
        "selected_tools": ",".join(decision["selected_tools"]),
        "validation_issues": " | ".join(decision["validation_issues"]),
    })
routes = pd.DataFrame(route_rows)
check("Every study routed", len(routes) == EXPECTED[MODE][0])
# A block is allowed only for QC/anatomy reasons; any other block would mean the cell's rule
# and agent_policy v1.1 disagree (a code error, not a property of the data).
clean = (routes["qc_status"] == "PASS") & (routes["anatomy_status"] == "SUPPORTED_WRIST")
check("Policy accepts every clean record", (routes.loc[clean, "policy_status"] == "SUCCESS").all(),
      routes.loc[clean & (routes["policy_status"] != "SUCCESS"), "validation_issues"].head(3).tolist())

# ---- 4. Save routes -----------------------------------------------------------------------
routes.to_csv(RUN_DIR / "study_routes.csv", index=False)
RUN["predictions_sha256"]["study_routes.csv"] = sha256(RUN_DIR / "study_routes.csv")
RUN["routes_saved_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
print("Routes:", routes["route"].value_counts().to_dict(), "| QC:", routes["qc_status"].value_counts().to_dict())
print("\nCELL 4: PASSED ->", RUN_DIR / "study_routes.csv")

PASS  agent_policy version 1.1  1.1
PASS  QC result for every image  
PASS  Every study routed  
PASS  Policy accepts every clean record  []
Routes: {'priority_human_review': 490, 'routine_workflow': 200, 'SAFETY_BLOCK': 1} | QC: {'PASS': 690, 'WARNING': 1}

CELL 4: PASSED -> /content/drive/MyDrive/DATA/OFFICIAL_VALID_FINAL/internal_20261110T084208Z/study_routes.csv


In [5]:
# =====================================================================================
# CELL 5 — Metrics with 95 % confidence intervals and the run summary
# Purpose : compute the endpoints, fixed before the run, once:
#             model level   : sensitivity and FN (primary); specificity, FP, flagged share,
#                             AUROC, AUPRC
#             workflow level: priority share, priority by view disagreement only, routine FN,
#                             abnormal studies outside priority, SAFETY_BLOCK count
# CIs     : patient-cluster bootstrap as in notebook 06 (bootstrap_interval): 1000 replicates,
#           seed 42, threshold FIXED, percentile 2.5/97.5
#           (all studies of a sampled patient enter together; one-class replicates skipped).
# Check   : AUROC, AUPRC, FN, FP recomputed independently with scikit-learn / plain counts.
# Writes  : summary.json (written once; in MODE "official" a second summary is refused).
# MODE    : "internal" -> reference values of the dry run; "official" -> the single run.
# =====================================================================================
from sklearn.metrics import roc_auc_score, average_precision_score
from src.evaluation.model_eval import compute_study_metrics

N_BOOT, SEED = 1000, 42   # as in notebook 06: bootstrap_interval(seed=42, iterations=1000)
SUMMARY_PATH = RUN_DIR / "summary.json"
if MODE == "official":   # last guard right before labels are used for metrics
    previous = sorted(OUT_ROOT.glob("official_*/summary.json"))
    check("Still no previous official run", not previous, ", ".join(map(str, previous)))
check("summary.json not written yet", not SUMMARY_PATH.exists())

# ---- 1. One table: label, study probability, route ------------------------------------------
data = studies.merge(routes, on="study_id", how="inner", validate="one_to_one")
check("Labels joined for every study", len(data) == EXPECTED[MODE][0])
y = data["label"].to_numpy(int)
p = data["abnormal_probability"].to_numpy(float)
route = data["route"].to_numpy(str)
patients = data["patient_id"].to_numpy(str)


def endpoints(idx):
    """All endpoints for the studies in idx (indices may repeat in the bootstrap)."""
    m = compute_study_metrics(y[idx], p[idx], threshold=THRESHOLD)   # repository metric code
    yy, pp, rr = y[idx], p[idx], route[idx]
    priority = rr == "priority_human_review"
    return {
        "sensitivity": m["sensitivity"], "fn": m["confusion_matrix"]["fn"],
        "specificity": m["specificity"], "fp": m["confusion_matrix"]["fp"],
        "flagged_share": float((pp >= THRESHOLD).mean()), "auroc": m["auroc"], "auprc": m["auprc"],
        "priority_share": float(priority.mean()),
        "priority_by_disagreement_only": int((priority & (pp < THRESHOLD)).sum()),
        "routine_fn": int(((rr == "routine_workflow") & (yy == 1)).sum()),
        "abnormal_outside_priority": int((~priority & (yy == 1)).sum()),
        "safety_block": int((rr == "SAFETY_BLOCK").sum()),
    }


# ---- 2. Point estimates and an independent cross-check ---------------------------------------
point = endpoints(np.arange(len(y)))
check("AUROC = sklearn", abs(point["auroc"] - roc_auc_score(y, p)) < 1e-9)
check("AUPRC = sklearn", abs(point["auprc"] - average_precision_score(y, p)) < 1e-9)
check("FN and FP = plain counts", point["fn"] == int(((p < THRESHOLD) & (y == 1)).sum())
      and point["fp"] == int(((p >= THRESHOLD) & (y == 0)).sum()))

# ---- 3. Patient-cluster bootstrap, threshold fixed ---------------------------------------------
rng = np.random.default_rng(SEED)
unique_patients = np.unique(patients)
rows_of = {pt: np.flatnonzero(patients == pt) for pt in unique_patients}
replicates, skipped = [], 0
for _ in range(N_BOOT):
    idx = np.concatenate([rows_of[pt] for pt in rng.choice(unique_patients, size=len(unique_patients), replace=True)])
    if len(np.unique(y[idx])) < 2:
        skipped += 1
        continue
    replicates.append(endpoints(idx))
boot = pd.DataFrame(replicates)
CI_KEYS = ["sensitivity", "specificity", "flagged_share", "auroc", "auprc", "priority_share"]
ci = {k: [float(boot[k].quantile(0.025)), float(boot[k].quantile(0.975))] for k in CI_KEYS}

# ---- 4. Reading of the primary endpoint (rule fixed before the run) --------------------------
if ci["sensitivity"][1] < 0.95:
    reading = "sensitivity target >= 0.95 NOT reproduced (upper 95% CI bound < 0.95)"
elif point["sensitivity"] >= 0.95:
    reading = "sensitivity target reproduced in the point estimate"
else:
    reading = "compatible with the target, but below it in the point estimate"

# ---- 5. Summary (aggregate numbers only) ---------------------------------------------------------
RUN.update({
    "population": {"studies": int(len(y)), "abnormal": int(y.sum()), "normal": int((y == 0).sum()),
                   "patients": int(len(unique_patients)),
                   "studies_within_0.001_of_threshold": int((np.abs(p - THRESHOLD) <= 1e-3).sum())},
    "endpoints": point, "ci95": ci,
    "bootstrap": {"replicates": N_BOOT, "seed": SEED, "skipped_one_class": skipped, "unit": "patient",
                  "threshold": "fixed", "method": "as in notebook 06 bootstrap_interval"},
    "primary_reading": reading,
    "route_counts": routes["route"].value_counts().to_dict(),
    "qc_counts": routes["qc_status"].value_counts().to_dict(),
    "summary_saved_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
})
SUMMARY_PATH.write_text(json.dumps(RUN, indent=1, default=str))
table = pd.DataFrame({"point": pd.Series(point)})
table["ci95"] = [f"[{ci[k][0]:.3f}, {ci[k][1]:.3f}]" if k in ci else "" for k in table.index]
print(table.to_string(float_format=lambda v: f"{v:.4f}"))
print("\nPopulation:", RUN["population"])
print("Bootstrap: skipped one-class replicates =", skipped)
print("Primary reading:", reading)
print("\nCELL 5: PASSED ->", SUMMARY_PATH)

PASS  summary.json not written yet  
PASS  Labels joined for every study  
PASS  AUROC = sklearn  
PASS  AUPRC = sklearn  
PASS  FN and FP = plain counts  
                                 point            ci95
sensitivity                     0.9480  [0.920, 0.972]
fn                             14.0000                
specificity                     0.5829  [0.536, 0.628]
fp                            176.0000                
flagged_share                   0.6237  [0.590, 0.658]
auroc                           0.9128  [0.889, 0.935]
auprc                           0.8781  [0.832, 0.916]
priority_share                  0.7091  [0.677, 0.744]
priority_by_disagreement_only  59.0000                
routine_fn                     10.0000                
abnormal_outside_priority      10.0000                
safety_block                    1.0000                

Population: {'studies': 691, 'abnormal': 269, 'normal': 422, 'patients': 653, 'studies_within_0.001_of_threshold': 1}
Bootstrap: